# Tutorial 11 - Spin-Triplet Correlations

**Goal:** Generate and visualize long-range equal-spin triplet superconductivity in a magnetic multilayer.

When adjacent ferromagnetic layers have non-collinear magnetization,
singlet Cooper pairs are converted to equal-spin triplets at interfaces.
These triplets decay over the long-range coherence length $\xi_N \gg \xi_F$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import supermag

supermag.apply_theme("publication")


## 1. Collinear vs non-collinear bilayer

Compare a collinear bilayer ($\alpha = 0$) with a non-collinear bilayer ($\alpha = \pi/2$).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, angle, label in zip(
    axes, [0.0, np.pi / 2], ["collinear", "non-collinear"]
):
    x, f_t = supermag.triplet.solve(
        n_layers=2,
        thicknesses=[10.0, 10.0],
        magnetization_angles=[0.0, angle],
        xi_F=1.0, xi_N=10.0, n_grid=400,
    )
    ax.plot(x, f_t, "k-", lw=2)
    ax.axvline(10.0, ls=":", color="gray", alpha=0.5)
    ax.set_xlabel("x (nm)")
    ax.set_title(label)
axes[0].set_ylabel(r"$|f_{\uparrow\uparrow}(x)|$")
plt.tight_layout()


## 2. Misalignment-angle dependence

Triplet amplitude follows a $\sin(\alpha)$ envelope.


In [ ]:
angles = np.linspace(0, np.pi, 50)
peak_amp = []
for alpha in angles:
    x, f_t = supermag.triplet.solve(
        n_layers=2,
        thicknesses=[10.0, 10.0],
        magnetization_angles=[0.0, alpha],
        xi_F=1.0, xi_N=10.0, n_grid=200,
    )
    peak_amp.append(np.max(f_t))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.degrees(angles), peak_amp, "ko-", ms=3)
sin_env = np.sin(angles) * np.max(peak_amp)
ax.plot(np.degrees(angles), sin_env, "C3--", label=r"$\sin(\alpha)$ envelope")
ax.set_xlabel(r"Misalignment angle $\alpha$ (deg)")
ax.set_ylabel(r"Peak $|f_{\uparrow\uparrow}|$")
ax.legend()
ax.set_title("Triplet generation vs misalignment")
plt.tight_layout()


## 3. Domain wall as a triplet generator

Model a 180-degree domain wall as many thin layers with rotating magnetization.


In [ ]:
N_wall = 10
wall_thickness = 20.0
layer_d = wall_thickness / N_wall
thicknesses = [layer_d] * N_wall
angles_wall = np.linspace(0, np.pi, N_wall)

x, f_t = supermag.triplet.solve(
    n_layers=N_wall,
    thicknesses=thicknesses,
    magnetization_angles=angles_wall,
    xi_F=1.0, xi_N=8.0, n_grid=500,
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True, gridspec_kw={"height_ratios": [1, 2]})
wall_x = np.cumsum([0] + thicknesses[:-1]) + layer_d / 2
ax1.plot(wall_x, np.degrees(angles_wall), "C0o-")
ax1.set_ylabel(r"$\theta(x)$ (deg)")
ax1.set_title("Domain wall rotation and triplet amplitude")

ax2.plot(x, f_t, "k-", lw=2)
for xi in np.cumsum(thicknesses[:-1]):
    ax2.axvline(xi, ls=":", color="gray", alpha=0.3)
ax2.set_xlabel("x (nm)")
ax2.set_ylabel(r"$|f_{\uparrow\uparrow}(x)|$")
plt.tight_layout()


## 4. Domain wall width effect

Peak triplet amplitude depends on wall width relative to $\xi_N$.
Narrow walls keep many interfaces within one coherence length, increasing the peak.


In [ ]:
widths = np.linspace(5, 50, 30)
N_sub = 10
peak_width = []
for w in widths:
    d_sub = w / N_sub
    x, f_t = supermag.triplet.solve(
        n_layers=N_sub,
        thicknesses=[d_sub] * N_sub,
        magnetization_angles=np.linspace(0, np.pi, N_sub),
        xi_F=1.0, xi_N=10.0, n_grid=300,
    )
    peak_width.append(np.max(f_t))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(widths, peak_width, "ko-", ms=4)
ax.set_xlabel("Domain wall width (nm)")
ax.set_ylabel(r"Peak $|f_{\uparrow\uparrow}|$")
ax.set_title("Triplet peak vs domain wall width")
plt.tight_layout()


## 5. Triplet spin valve

Use a 3-layer valve: F1 / F_rot / F1. Rotating the middle layer turns the triplet channel on/off.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for angle_mid, ls, label in [(0.0, "--", "parallel"), (np.pi / 4, "-.", "45 deg"), (np.pi / 2, "-", "90 deg")]:
    x, f_t = supermag.triplet.solve(
        n_layers=3,
        thicknesses=[5.0, 10.0, 5.0],
        magnetization_angles=[0.0, angle_mid, 0.0],
        xi_F=1.0, xi_N=10.0, n_grid=500,
    )
    ax.plot(x, f_t, ls, lw=2, label=f"middle = {label}")

for boundary in [5.0, 15.0]:
    ax.axvline(boundary, ls=":", color="gray", alpha=0.5)
ylim = ax.get_ylim()
ax.text(2.5, ylim[1] * 0.9, r"F$_1$", ha="center", fontsize=11)
ax.text(10.0, ylim[1] * 0.9, r"F$_{\mathrm{rot}}$", ha="center", fontsize=11)
ax.text(17.5, ylim[1] * 0.9, r"F$_1$", ha="center", fontsize=11)
ax.set_xlabel("x (nm)")
ax.set_ylabel(r"$|f_{\uparrow\uparrow}(x)|$")
ax.legend()
ax.set_title(r"Triplet spin valve: F$_1$ / F$_{\mathrm{rot}}$ / F$_1$")
plt.tight_layout()


## Summary

- Triplet amplitude follows $\sin(\alpha)$ and is maximal near $\alpha = \pi/2$.
- Domain walls create distributed triplet generation at many interfaces.
- Narrower walls (relative to $\xi_N$) give higher peak triplet amplitude.
- The triplet spin valve switches on with middle-layer rotation.
